# HbA1c Disease Classifier — Model 2: Hierarchical Disease Classifier (Diabetes)

Optimized for 3M diabetic claims. Standalone script that loads data, builds and trains
the hierarchical multi-task classifier for diabetes diagnosis classification,
evaluates, and saves all artifacts.

In [1]:
# ==================== IMPORTS & SETUP ====================

import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from datetime import datetime
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 1. Anchor to the project root (bypassing the app/resultshield_lite nesting)
BASE_DIR = Path.home() / "localfiles" / "lstm"

# 2. Main Directory Paths
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "visualizations" / "plots"

# 3. Specific File Paths for HbA1c Model
DATA_PATH = DATA_DIR / "kenyan_diabetic_claims_20260321_145103.csv"

# 4. Ensure directories exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# 5. Global Toggle - Set to False for full 3M training
TEST_MODE = False  # False = Full training on 3M rows

# 6. Sampling parameters (only used if TEST_MODE = True)
SAMPLE_SIZE = 80 if TEST_MODE else None

print("=" * 60)
print("  HbA1c Disease Classifier — Model 2 (Diabetes)")
print("=" * 60)
print(f"  TensorFlow : {tf.__version__}")
print(f"  Data       : {DATA_PATH}")
print(f"  Models dir : {MODEL_DIR}")
print(f"  Plots dir  : {PLOTS_DIR}")
print(f"  Test Mode  : {TEST_MODE}")
if TEST_MODE:
    print(f"  Sample Size: {SAMPLE_SIZE}")
else:
    print(f"  Mode       : FULL TRAINING (3M rows)")
print("=" * 60)

2026-03-21 16:08:39.606728: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 16:08:39.672899: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 16:08:39.672945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 16:08:39.674269: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 16:08:39.682834: I tensorflow/core/platform/cpu_feature_guar

  HbA1c Disease Classifier — Model 2 (Diabetes)
  TensorFlow : 2.15.0
  Data       : /home/azureuser/localfiles/lstm/data/kenyan_diabetic_claims_20260321_145103.csv
  Models dir : /home/azureuser/localfiles/lstm/models
  Plots dir  : /home/azureuser/localfiles/lstm/visualizations/plots
  Test Mode  : False
  Mode       : FULL TRAINING (3M rows)


In [2]:
# ==================== LOAD DATA ====================

print("\n📂 Loading dataset...")
df_full = pd.read_csv(DATA_PATH)
print(f"  Total claims in dataset: {len(df_full):,}")

# Convert dates if present
if 'date' in df_full.columns:
    df_full['date'] = pd.to_datetime(df_full['date'], errors='coerce')

# Drop rows with missing diagnosis
df_full = df_full.dropna(subset=['diagnosis']).reset_index(drop=True)

# Ensure disease_category is consistent
df_full['disease_category'] = 'diabetes'

# For test mode, use sample; for full training, use all data
if TEST_MODE:
    df = df_full.sample(n=min(SAMPLE_SIZE, len(df_full)), random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"\n🧪 TEST MODE: Using {len(df)} samples")
else:
    df = df_full.copy()
    print(f"\n🚀 FULL TRAINING MODE: Using {len(df):,} claims")

print(f"\n  Diabetes subtypes:")
print(df['diagnosis'].value_counts())


📂 Loading dataset...
  Total claims in dataset: 3,000,000

🚀 FULL TRAINING MODE: Using 3,000,000 claims

  Diabetes subtypes:
Diabetic Nephropathy    1000000
Diabetes - (DKA/HHS)    1000000
Diabetic Infections     1000000
Name: diagnosis, dtype: int64


In [3]:
# ==================== FEATURE DEFINITION ====================

# HbA1c specific features
LAB_FEATURES = ['HBA1C', 'CREATININE', 'UREA']
ALL_FEATURES = LAB_FEATURES
N_FEATURES = len(ALL_FEATURES)

# Handle missing values
df[ALL_FEATURES] = df[ALL_FEATURES].fillna(df[ALL_FEATURES].median())

X_raw = df[ALL_FEATURES].values

print(f"\n📊 Input features ({N_FEATURES}): {ALL_FEATURES}")
print(f"  Feature statistics:")
print(df[ALL_FEATURES].describe())


📊 Input features (3): ['HBA1C', 'CREATININE', 'UREA']
  Feature statistics:
              HBA1C    CREATININE          UREA
count  3.000000e+06  3.000000e+06  3.000000e+06
mean   8.541798e+00  1.644000e+00  4.464026e+01
std    3.902673e+00  1.401391e+00  3.560239e+01
min    1.100000e+00  0.000000e+00  8.000000e-01
25%    5.300000e+00  7.000000e-01  1.620000e+01
50%    9.000000e+00  1.200000e+00  3.800000e+01
75%    1.140000e+01  2.300000e+00  6.400000e+01
max    1.800000e+01  7.500000e+00  1.800000e+02


In [4]:
# ==================== ENCODE TARGET VARIABLES ====================

# Display the diagnosis hierarchy
print("\n📊 Diabetes Subtypes Distribution:")
diagnosis_counts = df['diagnosis'].value_counts()
for diag, count in diagnosis_counts.items():
    print(f"  {diag}: {count} samples ({count/len(df):.1%})")

# 1. Encode disease category (primary task - always "diabetes" but kept for consistency)
category_encoder = LabelEncoder()
y_category = category_encoder.fit_transform(df['disease_category'])
NUM_CATEGORIES = len(category_encoder.classes_)

# One-hot encode for loss function
y_category_onehot = tf.keras.utils.to_categorical(y_category, num_classes=NUM_CATEGORIES)

print(f"\n🎯 Primary task: Disease Categories ({NUM_CATEGORIES})")
for i, cat in enumerate(category_encoder.classes_):
    count = (df['disease_category'] == cat).sum()
    print(f"  {i}: {cat} ({count:,} samples, {count/len(df):.1%})")

# 2. Encode specific diagnosis (secondary task - diabetes subtype)
diagnosis_encoder = LabelEncoder()
y_diagnosis = diagnosis_encoder.fit_transform(df['diagnosis'])
NUM_DIAGNOSES = len(diagnosis_encoder.classes_)

# Create diagnosis to category mapping (all map to diabetes)
diagnosis_to_category = {}
for diagnosis in df['diagnosis'].unique():
    diagnosis_to_category[diagnosis] = 'diabetes'

print(f"\n🎯 Secondary task: Specific Diagnoses ({NUM_DIAGNOSES})")
print(f"  All diagnoses map to category: diabetes")


📊 Diabetes Subtypes Distribution:
  Diabetic Nephropathy: 1000000 samples (33.3%)
  Diabetes - (DKA/HHS): 1000000 samples (33.3%)
  Diabetic Infections: 1000000 samples (33.3%)

🎯 Primary task: Disease Categories (1)
  0: diabetes (3,000,000 samples, 100.0%)

🎯 Secondary task: Specific Diagnoses (3)
  All diagnoses map to category: diabetes


In [5]:
# ==================== FEATURE SCALING ====================

# Use StandardScaler for classifier (better for neural networks)
scaler_model2 = StandardScaler()
X2_scaled = scaler_model2.fit_transform(X_raw)

# Save scaler
scaler2_path = MODEL_DIR / "hba1c_model2_scaler.pkl"
joblib.dump(scaler_model2, scaler2_path)
print(f"\n✅ Scaler saved to {scaler2_path}")


✅ Scaler saved to /home/azureuser/localfiles/lstm/models/hba1c_model2_scaler.pkl


In [6]:
# ==================== TRAIN/VAL/TEST SPLIT ====================

# Split data (maintain stratification on diagnosis)
X2_train, X2_temp, y_cat_train, y_cat_temp, y_diag_train, y_diag_temp = train_test_split(
    X2_scaled, y_category_onehot, y_diagnosis, 
    test_size=0.3, random_state=RANDOM_SEED, stratify=y_diagnosis
)

X2_val, X2_test, y_cat_val, y_cat_test, y_diag_val, y_diag_test = train_test_split(
    X2_temp, y_cat_temp, y_diag_temp,
    test_size=0.5, random_state=RANDOM_SEED, stratify=y_diag_temp
)

print(f"\n📊 Dataset Split:")
print(f"  Train set: {X2_train.shape}")
print(f"  Validation set: {X2_val.shape}")
print(f"  Test set: {X2_test.shape}")

print(f"\nDiagnosis distribution in train set:")
train_diag_counts = pd.Series(y_diag_train).value_counts()
for idx in train_diag_counts.index[:10]:  # Show top 10
    diag_name = diagnosis_encoder.inverse_transform([idx])[0]
    print(f"  {diag_name}: {train_diag_counts[idx]} samples ({train_diag_counts[idx]/len(y_diag_train):.1%})")


📊 Dataset Split:
  Train set: (2100000, 3)
  Validation set: (450000, 3)
  Test set: (450000, 3)

Diagnosis distribution in train set:
  Diabetic Infections: 700000 samples (33.3%)
  Diabetic Nephropathy: 700000 samples (33.3%)
  Diabetes - (DKA/HHS): 700000 samples (33.3%)


In [7]:
# ==================== BUILD HIERARCHICAL CLASSIFIER (SCALED FOR 3M ROWS) ====================

def build_hierarchical_classifier(input_dim, num_categories, num_diagnoses, diagnosis_to_cat_map):
    """
    Build a neural network for diabetes diagnosis classification with:
    - Increased capacity for 3M rows
    - Shared layers for feature extraction
    - Category prediction head (always diabetes)
    - Diagnosis prediction head (diabetes subtype)
    """
    
    # Input layer
    inputs = layers.Input(shape=(input_dim,), name='input')
    
    # ===== SHARED FEATURE EXTRACTION (Scaled for 3M rows) =====
    # First block - wider layers for better feature learning
    x = layers.Dense(128, activation='relu', name='shared_dense1')(inputs)
    x = layers.BatchNormalization(name='shared_bn1')(x)
    x = layers.Dropout(0.3, name='shared_dropout1')(x)
    
    x = layers.Dense(64, activation='relu', name='shared_dense2')(x)
    x = layers.BatchNormalization(name='shared_bn2')(x)
    x = layers.Dropout(0.3, name='shared_dropout2')(x)
    
    x = layers.Dense(32, activation='relu', name='shared_dense3')(x)
    x = layers.BatchNormalization(name='shared_bn3')(x)
    x = layers.Dropout(0.2, name='shared_dropout3')(x)
    
    # ===== TASK 1: Category Prediction =====
    category_features = layers.Dense(16, activation='relu', name='category_dense')(x)
    category_features = layers.BatchNormalization(name='category_bn')(category_features)
    category_output = layers.Dense(num_categories, activation='softmax', name='category_output')(category_features)
    
    # ===== TASK 2: Diagnosis Prediction =====
    # Additional layers specific to diagnosis
    diagnosis_features = layers.Concatenate(name='diagnosis_concat')([x, category_features])
    
    diagnosis_features = layers.Dense(64, activation='relu', name='diagnosis_dense1')(diagnosis_features)
    diagnosis_features = layers.BatchNormalization(name='diagnosis_bn1')(diagnosis_features)
    diagnosis_features = layers.Dropout(0.3, name='diagnosis_dropout1')(diagnosis_features)
    
    diagnosis_features = layers.Dense(32, activation='relu', name='diagnosis_dense2')(diagnosis_features)
    diagnosis_features = layers.BatchNormalization(name='diagnosis_bn2')(diagnosis_features)
    diagnosis_features = layers.Dropout(0.3, name='diagnosis_dropout2')(diagnosis_features)
    
    diagnosis_features = layers.Dense(16, activation='relu', name='diagnosis_dense3')(diagnosis_features)
    diagnosis_features = layers.BatchNormalization(name='diagnosis_bn3')(diagnosis_features)
    
    # Diagnosis output (full softmax over all diabetes subtypes)
    diagnosis_output = layers.Dense(num_diagnoses, activation='softmax', name='diagnosis_output')(diagnosis_features)
    
    # Create model
    model = models.Model(
        inputs=inputs, 
        outputs=[category_output, diagnosis_output],
        name='hba1c_disease_classifier'
    )
    
    return model

# Build model
model2 = build_hierarchical_classifier(
    N_FEATURES, 
    NUM_CATEGORIES, 
    NUM_DIAGNOSES,
    diagnosis_to_category
)

2026-03-21 16:10:22.720798: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [8]:
# ==================== CUSTOM LOSS WEIGHTS ====================

# Weight the losses (category is always diabetes, so lower weight)
loss_weights = {
    'category_output': 0.2,    # 20% weight on category (trivial for diabetes)
    'diagnosis_output': 0.8     # 80% weight on diagnosis (main task)
}

# Compile with multiple losses
model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'category_output': 'categorical_crossentropy',
        'diagnosis_output': 'sparse_categorical_crossentropy'
    },
    loss_weights=loss_weights,
    metrics={
        'category_output': ['accuracy', tf.keras.metrics.AUC(name='auc')],
        'diagnosis_output': ['accuracy']
    }
)

model2.summary()

# Plot model architecture
try:
    tf.keras.utils.plot_model(
        model2,
        to_file=PLOTS_DIR / "hba1c_model2_architecture.png",
        show_shapes=True,
        show_layer_names=True,
        dpi=100
    )
    print(f"\n✅ Model architecture saved to {PLOTS_DIR}/hba1c_model2_architecture.png")
except Exception as e:
    print(f"\n⚠️  plot_model skipped: {e}")

Model: "hba1c_disease_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 3)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense1       │ (None, 128)       │        512 │ input[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn1          │ (None, 128)       │        512 │ shared_dense1[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout1     │ (None, 128)       │          0 │ shared_bn1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense2       │ (None, 64)        │      8,256 │ shared_dropout1[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn2          │ (None, 64)        │        256 │ shared_dense2[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout2     │ (None, 64)        │          0 │ shared_bn2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense3       │ (None, 32)        │      2,080 │ shared_dropout2[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bn3          │ (None, 32)        │        128 │ shared_dense3[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout3     │ (None, 32)        │          0 │ shared_bn3[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_dense      │ (None, 16)        │        528 │ shared_dropout3[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_bn         │ (None, 16)        │         64 │ category_dense[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diagnosis_concat    │ (None, 48)        │          0 │ shared_dropout3[… │
│ (Concatenate)       │                   │            │ category_bn[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diagnosis_dense1    │ (None, 64)        │      3,136 │ diagnosis_concat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diagnosis_bn1       │ (None, 64)        │        256 │ diagnosis_dense1… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diagnosis_dropout1  │ (None, 64)        │          0 │ diagnosis_bn1[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diagnosis_dense2    │ (None, 32)        │      2,080 │ diagnosis_dropou… │
│ (Dense)             │                   │            │                 

 Total params: 18,596 (72.64 KB)

 Trainable params: 17,892 (69.89 KB)

 Non-trainable params: 704 (2.75 KB)

You must install pydot (`pip install pydot`) for `plot_model` to work.

✅ Model architecture saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model2_architecture.png


In [9]:
# ==================== TRAIN HIERARCHICAL CLASSIFIER (OPTIMIZED FOR 3M ROWS) ====================

# Training parameters optimized for 3M rows
EPOCHS_CLF = 30 if TEST_MODE else 80          # More epochs for convergence on 3M
BATCH_SIZE_CLF = 16 if TEST_MODE else 1024     # Large batches for 3M rows

# Callbacks
model2_path = MODEL_DIR / "hba1c_model2_classifier.keras"

callbacks2 = [
    tf.keras.callbacks.ModelCheckpoint(
        model2_path,
        monitor='val_diagnosis_output_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,                            # More patience for large dataset
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,                             # Wait longer before reducing LR
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        MODEL_DIR / "hba1c_model2_training_log.csv"
    )
]

print("\n🚀 Starting Model 2 (HbA1c Hierarchical Classifier) training...\n")
print(f"Training Configuration:")
print(f"  - Epochs: {EPOCHS_CLF}")
print(f"  - Batch size: {BATCH_SIZE_CLF}")
print(f"  - Training samples: {len(X2_train):,}")
print(f"  - Validation samples: {len(X2_val):,}")
print(f"  - Early stopping patience: 10")
print(f"  - Learning rate: 0.001 (with ReduceLROnPlateau)")

history2 = model2.fit(
    X2_train, 
    {
        'category_output': y_cat_train,
        'diagnosis_output': y_diag_train
    },
    validation_data=(
        X2_val,
        {
            'category_output': y_cat_val,
            'diagnosis_output': y_diag_val
        }
    ),
    epochs=EPOCHS_CLF,
    batch_size=BATCH_SIZE_CLF,
    callbacks=callbacks2,
    verbose=1,
    shuffle=True
)

print(f"\n✅ Model 2 training complete!")
if 'val_diagnosis_output_accuracy' in history2.history:
    best_acc = max(history2.history['val_diagnosis_output_accuracy'])
    print(f"Best val_diagnosis_accuracy: {best_acc:.4f}")


🚀 Starting Model 2 (HbA1c Hierarchical Classifier) training...

Training Configuration:
  - Epochs: 80
  - Batch size: 1024
  - Training samples: 2,100,000
  - Validation samples: 450,000
  - Early stopping patience: 10
  - Learning rate: 0.001 (with ReduceLROnPlateau)
Epoch 1/80
2047/2051 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - category_output_accuracy: 1.0000 - category_output_auc: 0.0000e+00 - category_output_loss: 0.0000e+00 - diagnosis_output_accuracy: 0.4694 - diagnosis_output_loss: 1.0223 - loss: 0.8178
Epoch 1: val_diagnosis_output_accuracy improved from None to 0.50060, saving model to /home/azureuser/localfiles/lstm/models/hba1c_model2_classifier.keras
2051/2051 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - category_output_accuracy: 1.0000 - category_output_auc: 0.0000e+00 - category_output_loss: 0.0000e+00 - diagnosis_output_accuracy: 0.4848 - diagnosis_output_loss: 0.9620 - loss: 0.7696 - val_category_output_accuracy: 1.0000 - val_category_output_auc: 0.0000e+00 - val_category_output_l

In [10]:
# ==================== PLOT TRAINING HISTORY ====================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Total Loss
ax = axes[0, 0]
ax.plot(history2.history['loss'], label='Training Loss', linewidth=2)
ax.plot(history2.history['val_loss'], label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Category Accuracy
ax = axes[0, 1]
ax.plot(history2.history['category_output_accuracy'], label='Training', linewidth=2)
ax.plot(history2.history['val_category_output_accuracy'], label='Validation', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Category Prediction Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Diagnosis Accuracy
ax = axes[1, 0]
ax.plot(history2.history['diagnosis_output_accuracy'], label='Training', linewidth=2)
ax.plot(history2.history['val_diagnosis_output_accuracy'], label='Validation', linewidth=2)
if 'val_diagnosis_output_accuracy' in history2.history:
    best_epoch = np.argmax(history2.history['val_diagnosis_output_accuracy'])
    ax.scatter(best_epoch, history2.history['val_diagnosis_output_accuracy'][best_epoch], 
              color='red', s=100, zorder=5,
              label=f'Best: {history2.history["val_diagnosis_output_accuracy"][best_epoch]:.3f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Diagnosis Prediction Accuracy (Diabetes Subtype)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Category AUC
ax = axes[1, 1]
ax.plot(history2.history['category_output_auc'], label='Training AUC', linewidth=2)
ax.plot(history2.history['val_category_output_auc'], label='Validation AUC', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.set_title('Category Prediction AUC')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Model 2: HbA1c Hierarchical Classifier Training History', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
plot_path = PLOTS_DIR / "hba1c_model2_training_history.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Plot saved to {plot_path}")

✅ Plot saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model2_training_history.png


In [11]:
# ==================== EVALUATE CLASSIFIER ====================

# Predict on test set
y_pred_cat_proba, y_pred_diag_proba = model2.predict(X2_test, verbose=0)
y_pred_cat = np.argmax(y_pred_cat_proba, axis=1)
y_pred_diag = np.argmax(y_pred_diag_proba, axis=1)

y_true_cat = np.argmax(y_cat_test, axis=1)
y_true_diag = y_diag_test

# ===== Category Evaluation =====
print("\n" + "="*60)
print("📊 CATEGORY PREDICTION EVALUATION")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_true_cat, y_pred_cat, target_names=category_encoder.classes_))

# Category confusion matrix
cm_cat = confusion_matrix(y_true_cat, y_pred_cat)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_cat, annot=True, fmt='d', cmap='Blues', 
            xticklabels=category_encoder.classes_, 
            yticklabels=category_encoder.classes_)
plt.title('HbA1c Model: Category Confusion Matrix')
plt.ylabel('True Category')
plt.xlabel('Predicted Category')
plt.tight_layout()

# Save
plot_path = PLOTS_DIR / "hba1c_model2_category_confusion.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Category plot saved to {plot_path}")

# ===== Diagnosis Evaluation =====
print("\n" + "="*60)
print("📊 DIAGNOSIS PREDICTION EVALUATION")
print("="*60)
print(f"Overall Diagnosis Accuracy: {np.mean(y_pred_diag == y_true_diag):.4f}")

# Per-diagnosis accuracy
print("\nDiagnosis Accuracy by Diabetes Subtype:")
for i, diag_name in enumerate(diagnosis_encoder.classes_):
    diag_indices = np.where(y_true_diag == i)[0]
    if len(diag_indices) > 0:
        diag_accuracy = np.mean(y_pred_diag[diag_indices] == i)
        print(f"  {diag_name}: {diag_accuracy:.4f} ({len(diag_indices)} samples)")


📊 CATEGORY PREDICTION EVALUATION

Classification Report:
              precision    recall  f1-score   support

    diabetes       1.00      1.00      1.00    450000

    accuracy                           1.00    450000
   macro avg       1.00      1.00      1.00    450000
weighted avg       1.00      1.00      1.00    450000

✅ Category plot saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model2_category_confusion.png

📊 DIAGNOSIS PREDICTION EVALUATION
Overall Diagnosis Accuracy: 0.5146

Diagnosis Accuracy by Diabetes Subtype:
  Diabetes - (DKA/HHS): 0.3485 (150000 samples)
  Diabetic Infections: 0.8950 (150000 samples)
  Diabetic Nephropathy: 0.3002 (150000 samples)


In [12]:
# ==================== DIAGNOSIS CONFUSION MATRIX ====================

# Create confusion matrix for diagnoses
cm_diag = confusion_matrix(y_true_diag, y_pred_diag)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_diag, annot=True, fmt='d', cmap='Blues',
            xticklabels=diagnosis_encoder.classes_,
            yticklabels=diagnosis_encoder.classes_)
plt.title('HbA1c Model: Diabetes Subtype Confusion Matrix')
plt.ylabel('True Diagnosis')
plt.xlabel('Predicted Diagnosis')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save
plot_path = PLOTS_DIR / "hba1c_model2_diagnosis_confusion.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Diagnosis plot saved to {plot_path}")

✅ Diagnosis plot saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model2_diagnosis_confusion.png


In [13]:
# ==================== MISMATCH SCORE CALCULATION ====================

def calculate_mismatch_score(y_true_cat, y_true_diag, y_pred_cat_proba, y_pred_diag_proba, 
                             category_encoder, diagnosis_encoder, diagnosis_to_category):
    """
    Calculate mismatch score for a diabetes claim.
    For HbA1c, mismatches occur when lab values don't match the expected pattern.
    """
    # Get predicted class indices
    pred_cat_idx = np.argmax(y_pred_cat_proba)
    pred_diag_idx = np.argmax(y_pred_diag_proba)
    
    # Get predicted names
    pred_cat = category_encoder.inverse_transform([pred_cat_idx])[0]
    pred_diag = diagnosis_encoder.inverse_transform([pred_diag_idx])[0]
    
    # True names
    true_cat = category_encoder.inverse_transform([y_true_cat])[0]
    true_diag = diagnosis_encoder.inverse_transform([y_true_diag])[0]
    
    # Category match score (always 0 for diabetes)
    cat_match_score = 0 if pred_cat == true_cat else 1
    
    # Diagnosis match score
    diag_match_score = 0 if pred_diag == true_diag else 1
    
    # Category consistency (always consistent for diabetes)
    category_consistency_score = 0
    
    # Combined mismatch score
    mismatch_score = (0.2 * cat_match_score + 0.8 * diag_match_score)
    
    return {
        'mismatch_score': mismatch_score,
        'cat_match': cat_match_score,
        'diag_match': diag_match_score,
        'category_consistency': category_consistency_score,
        'true_category': true_cat,
        'true_diagnosis': true_diag,
        'predicted_category': pred_cat,
        'predicted_diagnosis': pred_diag,
        'category_confidence': float(y_pred_cat_proba[pred_cat_idx]),
        'diagnosis_confidence': float(y_pred_diag_proba[pred_diag_idx])
    }

# Test on a few samples
print("\n" + "="*60)
print("📊 MISMATCH SCORE EXAMPLES")
print("="*60)

for i in range(min(5, len(X2_test))):
    result = calculate_mismatch_score(
        y_true_cat[i], y_true_diag[i],
        y_pred_cat_proba[i], y_pred_diag_proba[i],
        category_encoder, diagnosis_encoder, diagnosis_to_category
    )
    
    print(f"\nSample {i+1}:")
    print(f"  True: {result['true_diagnosis']}")
    print(f"  Pred: {result['predicted_diagnosis']} (confidence: {result['diagnosis_confidence']:.3f})")
    print(f"  Mismatch Score: {result['mismatch_score']:.3f}")


📊 MISMATCH SCORE EXAMPLES

Sample 1:
  True: Diabetic Nephropathy
  Pred: Diabetic Infections (confidence: 0.417)
  Mismatch Score: 0.800

Sample 2:
  True: Diabetic Infections
  Pred: Diabetes - (DKA/HHS) (confidence: 0.637)
  Mismatch Score: 0.800

Sample 3:
  True: Diabetic Nephropathy
  Pred: Diabetic Infections (confidence: 0.335)
  Mismatch Score: 0.800

Sample 4:
  True: Diabetes - (DKA/HHS)
  Pred: Diabetic Infections (confidence: 0.441)
  Mismatch Score: 0.800

Sample 5:
  True: Diabetic Nephropathy
  Pred: Diabetic Infections (confidence: 0.566)
  Mismatch Score: 0.800


In [14]:
# ==================== SAVE MODEL METADATA ====================

# Create diagnosis-to-category mapping for inference
diagnosis_to_category_map = {}
for diagnosis in diagnosis_encoder.classes_:
    diagnosis_to_category_map[diagnosis] = 'diabetes'

metadata2 = {
    'model_name': 'Model 2: HbA1c Hierarchical Disease Classifier',
    'model_type': 'multi_task_neural_network',
    'test_type': 'HbA1c',
    'input_dim': N_FEATURES,
    'feature_names': ALL_FEATURES,
    'num_categories': NUM_CATEGORIES,
    'category_names': category_encoder.classes_.tolist(),
    'num_diagnoses': NUM_DIAGNOSES,
    'diagnosis_names': diagnosis_encoder.classes_.tolist(),
    'diagnosis_to_category': diagnosis_to_category_map,
    'loss_weights': loss_weights,
    'train_size': len(X2_train),
    'val_size': len(X2_val),
    'test_size': len(X2_test),
    'test_diagnosis_accuracy': float(np.mean(y_pred_diag == y_true_diag)),
    'training_date': datetime.now().isoformat(),
    'epochs_trained': len(history2.history['loss']),
    'batch_size': BATCH_SIZE_CLF,
    'learning_rate': 0.001
}

# Save metadata
meta2_path = MODEL_DIR / "hba1c_model2_classifier_meta.json"
with open(meta2_path, 'w') as f:
    json.dump(metadata2, f, indent=2)

# Save encoders
category_encoder_path = MODEL_DIR / "hba1c_model2_category_encoder.pkl"
diagnosis_encoder_path = MODEL_DIR / "hba1c_model2_diagnosis_encoder.pkl"
joblib.dump(category_encoder, category_encoder_path)
joblib.dump(diagnosis_encoder, diagnosis_encoder_path)

print(f"\n✅ Model saved to: {model2_path}")
print(f"✅ Metadata saved to: {meta2_path}")
print(f"✅ Category encoder saved to: {category_encoder_path}")
print(f"✅ Diagnosis encoder saved to: {diagnosis_encoder_path}")


✅ Model saved to: /home/azureuser/localfiles/lstm/models/hba1c_model2_classifier.keras
✅ Metadata saved to: /home/azureuser/localfiles/lstm/models/hba1c_model2_classifier_meta.json
✅ Category encoder saved to: /home/azureuser/localfiles/lstm/models/hba1c_model2_category_encoder.pkl
✅ Diagnosis encoder saved to: /home/azureuser/localfiles/lstm/models/hba1c_model2_diagnosis_encoder.pkl


In [15]:
# ==================== TEST MODEL LOADING ====================

print("\n🔍 Testing Model 2 loading and inference...")

# Load model
loaded_model2 = tf.keras.models.load_model(model2_path)
print("✅ Model loaded successfully")

# Load metadata
with open(meta2_path, 'r') as f:
    loaded_meta2 = json.load(f)
print("✅ Metadata loaded successfully")

# Load scaler and encoders
loaded_scaler2 = joblib.load(scaler2_path)
loaded_category_encoder = joblib.load(category_encoder_path)
loaded_diagnosis_encoder = joblib.load(diagnosis_encoder_path)
print("✅ Scaler and encoders loaded successfully")

# Test prediction
test_samples2 = X2_scaled[:5]
cat_proba, diag_proba = loaded_model2.predict(test_samples2, verbose=0)

print(f"\n📊 Test inference:")
print(f"  Input shape: {test_samples2.shape}")
print(f"  Category output shape: {cat_proba.shape}")
print(f"  Diagnosis output shape: {diag_proba.shape}")

for i in range(min(5, len(test_samples2))):
    pred_cat_idx = np.argmax(cat_proba[i])
    pred_diag_idx = np.argmax(diag_proba[i])
    pred_cat = loaded_category_encoder.inverse_transform([pred_cat_idx])[0]
    pred_diag = loaded_diagnosis_encoder.inverse_transform([pred_diag_idx])[0]
    print(f"\n  Sample {i+1}:")
    print(f"    Predicted category: {pred_cat} (confidence: {cat_proba[i][pred_cat_idx]:.3f})")
    print(f"    Predicted diagnosis: {pred_diag} (confidence: {diag_proba[i][pred_diag_idx]:.3f})")

print("\n" + "="*60)
print("✅ HbA1c Disease Classifier Test Complete!")
print("="*60)


🔍 Testing Model 2 loading and inference...
✅ Model loaded successfully
✅ Metadata loaded successfully
✅ Scaler and encoders loaded successfully

📊 Test inference:
  Input shape: (5, 3)
  Category output shape: (5, 1)
  Diagnosis output shape: (5, 3)

  Sample 1:
    Predicted category: diabetes (confidence: 1.000)
    Predicted diagnosis: Diabetic Infections (confidence: 0.335)

  Sample 2:
    Predicted category: diabetes (confidence: 1.000)
    Predicted diagnosis: Diabetic Infections (confidence: 0.335)

  Sample 3:
    Predicted category: diabetes (confidence: 1.000)
    Predicted diagnosis: Diabetic Infections (confidence: 0.612)

  Sample 4:
    Predicted category: diabetes (confidence: 1.000)
    Predicted diagnosis: Diabetic Infections (confidence: 0.510)

  Sample 5:
    Predicted category: diabetes (confidence: 1.000)
    Predicted diagnosis: Diabetes - (DKA/HHS) (confidence: 0.995)

✅ HbA1c Disease Classifier Test Complete!
